In [1]:


import os
import shutil
import kagglehub

# 1. Download dataset using kagglehub
cache_path = kagglehub.dataset_download("blastchar/telco-customer-churn")

# 2. Ensure target directory exists
target_dir = "data/raw"
os.makedirs(target_dir, exist_ok=True)

# 3. Copy downloaded files from cache to project's data/raw folder
for file_name in os.listdir(cache_path):
    src_file = os.path.join(cache_path, file_name)
    dst_file = os.path.join(target_dir, file_name)
    if os.path.isfile(src_file):
        shutil.copy(src_file, dst_file)

print(f"Dataset downloaded and placed at: {os.path.abspath(target_dir)}")
print("Files inside data/raw:", os.listdir(target_dir))

✅ Dataset downloaded and placed at: C:\Users\waghm\MY PROJECTS\Telco Churn Predictor\data\raw
📂 Files inside data/raw: ['WA_Fn-UseC_-Telco-Customer-Churn.csv']


In [2]:
import os

old_path = "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
new_path = "data/raw/telco_churn.csv"

# Check if the original file exists before renaming
if os.path.exists(old_path):
    os.rename(old_path, new_path)
    print(f"File successfully renamed to: {new_path}")
else:
    print("Original file not found (it might already have been renamed).")

# Verify the updated contents of data/raw
print("Files inside data/raw:", os.listdir("data/raw"))

✅ File successfully renamed to: data/raw/telco_churn.csv
📂 Files inside data/raw: ['telco_churn.csv']


In [3]:
from pathlib import Path

def display_tree(dir_path: Path, prefix: str = "", ignore: set = None):
    if ignore is None:
        ignore = {".git", "__pycache__", ".ipynb_checkpoints", ".pytest_cache", "venv"}
    
    contents = [p for p in dir_path.iterdir() if p.name not in ignore]
    contents.sort(key=lambda p: (p.is_file(), p.name.lower()))
    
    pointers = ["├── "] * (len(contents) - 1) + ["└── "]
    for pointer, path in zip(pointers, contents):
        print(f"{prefix}{pointer}{path.name}")
        if path.is_dir():
            extension = "│   " if pointer == "├── " else "    "
            display_tree(path, prefix=prefix + extension, ignore=ignore)

print(f"📁 {Path.cwd().name}/")
display_tree(Path.cwd())

📁 Telco Churn Predictor/
├── .github
│   └── workflows
├── artifacts
├── configs
├── data
│   ├── external
│   ├── processed
│   └── raw
│       └── telco_churn.csv
├── docker
├── great_expectations
├── mlruns
│   ├── .trash
│   ├── 225977856873923587
│   │   ├── 7691d96668ba40b49a6098eec048a233
│   │   │   ├── artifacts
│   │   │   ├── metrics
│   │   │   │   ├── f1
│   │   │   │   ├── precision
│   │   │   │   ├── pred_time_sec
│   │   │   │   ├── recall
│   │   │   │   ├── roc_auc
│   │   │   │   └── train_time_sec
│   │   │   ├── outputs
│   │   │   │   └── m-5c28683daae445cea808b85abbe9c47e
│   │   │   │       └── meta.yaml
│   │   │   ├── params
│   │   │   │   ├── colsample_bytree
│   │   │   │   ├── eval_metric
│   │   │   │   ├── gamma
│   │   │   │   ├── learning_rate
│   │   │   │   ├── max_depth
│   │   │   │   ├── min_child_weight
│   │   │   │   ├── n_estimators
│   │   │   │   ├── n_jobs
│   │   │   │   ├── prediction_threshold
│   │   │   │   ├── random_state
│   │   │ 

In [2]:
from pathlib import Path

required_files = [
    "data/raw/telco_churn.csv",
    "src/data/load_data.py",
    "src/data/preprocess_data.py",
    "src/features/build_features.py",
    "src/models/metrics.py",
    "src/models/tune.py",
    "src/models/train.py",
    "src/utils/tracking.py",
    "src/serving/predict.py",
]

missing = [f for f in required_files if not Path(f).exists()]
if missing:
    print(f"❌ Missing required files: {missing}")
else:
    print("✅ Project structure is complete and verified.")

✅ Project structure is complete and verified.


In [1]:
from pathlib import Path

# In Python scripts: uses __file__ | In Jupyter: falls back to current working directory
current_path = Path(__file__).resolve() if "__file__" in globals() else Path.cwd().resolve()

print(f"Current Path: {current_path}")
print(f"parents[0]:   {current_path.parents[0]}")
print(f"parents[1]:   {current_path.parents[1]}")
print(f"parents[2]:   {current_path.parents[2]}")

Current Path: C:\Users\waghm\MY PROJECTS\Telco Churn Predictor
parents[0]:   C:\Users\waghm\MY PROJECTS
parents[1]:   C:\Users\waghm
parents[2]:   C:\Users


In [2]:
%%writefile .dockerignore
.git
.gitignore
__pycache__
*.pyc
*.pyo
*.pyd
.pytest_cache
.venv
env/
venv/
.github

Overwriting .dockerignore


In [3]:
%%writefile Dockerfile
FROM python:3.11-slim AS base

ENV PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1 \
    MLFLOW_ALLOW_FILE_STORE=true

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    curl \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

# --- Target 1: FastAPI Backend ---
FROM base AS fastapi
EXPOSE 8000
CMD ["uvicorn", "src.app.main:app", "--host", "0.0.0.0", "--port", "8000"]

# --- Target 2: Streamlit Frontend ---
FROM base AS streamlit
EXPOSE 8501
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]

Overwriting Dockerfile


In [4]:
%%writefile docker-compose.yml
version: '3.8'

services:
  fastapi:
    build:
      context: .
      target: fastapi
    container_name: telco_fastapi_backend
    ports:
      - "8000:8000"
    volumes:
      - ./mlruns:/app/mlruns
    environment:
      - MLFLOW_ALLOW_FILE_STORE=true
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 10s
      timeout: 5s
      retries: 3

  streamlit:
    build:
      context: .
      target: streamlit
    container_name: telco_streamlit_frontend
    ports:
      - "8501:8501"
    environment:
      - FASTAPI_URL=http://fastapi:8000
    depends_on:
      fastapi:
        condition: service_healthy

Overwriting docker-compose.yml


In [5]:
%%writefile app.py
import os
import requests
import streamlit as st

FASTAPI_URL = os.getenv("FASTAPI_URL", "http://localhost:8000")

st.set_page_config(page_title="Telco Churn Predictor", page_icon="🔮", layout="wide")
st.title("🔮 Telco Customer Churn Predictor")

# Backend Connection Check
try:
    health_res = requests.get(f"{FASTAPI_URL}/health", timeout=3)
    if health_res.status_code == 200:
        st.success(f"Connected to FastAPI Backend ({FASTAPI_URL})")
    else:
        st.warning("Backend service degraded.")
except Exception as e:
    st.error(f"Cannot connect to FastAPI at {FASTAPI_URL}: {e}")

st.info("Interactive input forms and probability gauge charts will be wired here next.")

Overwriting app.py
